# Day 10 v2 — Gold: All API Marts (For Loop)

**Source:** `silver-volume/api/` (17 Silver Delta tables from Day 8 v4)
**Sink:** `gold-volume/api/` (5 Gold Delta marts)

| Gold Mart | Source tables | Grain | Key use case |
|---|---|---|---|
| `sessions_enriched` | sessions + customers + vehicles + chargers + stations + tariffs + payments | 1 row/session | Revenue, energy, customer behaviour |
| `revenue_by_station` | sessions_enriched | 1 row/station/day | Station performance dashboard |
| `customer_summary` | sessions_enriched + complaints | 1 row/customer | Customer lifetime value |
| `charger_utilisation` | sessions_enriched + maintenance_events | 1 row/charger/day | Operational efficiency |
| `maintenance_summary` | maintenance_events + employees + stations | 1 row/event | MTTR, fault analysis |

**All marts:** full overwrite in this v2 version.

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print('Imports OK')

In [ ]:
# Cell 2 — Constants
SILVER_API = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
GOLD_API   = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/api'
PIPELINE   = 'pl_gold_api_all_marts_v2'
RUN_TS     = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'Silver : {SILVER_API}')
print(f'Gold   : {GOLD_API}')
print(f'RUN_TS : {RUN_TS}')

In [ ]:
# Cell 3 — Read all Silver tables
def read_silver(name):
    return spark.read.format('delta').load(f'{SILVER_API}/{name}')

sessions           = read_silver('sessions')
customers          = read_silver('customers')
vehicles           = read_silver('vehicles')
chargers           = read_silver('chargers')
stations           = read_silver('stations')
tariffs            = read_silver('tariffs')
payments           = read_silver('payments')
complaints         = read_silver('complaints')
maintenance_events = read_silver('maintenance_events')
employees          = read_silver('employees')
energy_prices      = read_silver('energy_prices')

print('All Silver tables loaded')

In [ ]:
# Cell 4 — Build sessions_enriched (base mart used by downstream marts)

cust_dim = customers.select(
    'customer_id',
    F.concat_ws(' ', 'first_name', 'last_name').alias('customer_name'),
    F.col('email').alias('customer_email'),
    F.col('city').alias('customer_city'),
    F.col('state').alias('customer_state'),
    F.col('country').alias('customer_country'),
)
veh_dim = vehicles.select(
    'vehicle_id',
    F.col('make').alias('vehicle_make'),
    F.col('model').alias('vehicle_model'),
    F.col('year').alias('vehicle_year'),
    F.col('battery_capacity').alias('battery_capacity_kwh'),
    'range_km',
)
chg_dim = chargers.select(
    'charger_id',
    'charger_type',
    F.col('power_kw').alias('charger_power_kw'),
    'connector_type',
    F.col('status').alias('charger_status'),
)
sta_dim = stations.select(
    'station_id',
    F.col('name').alias('station_name'),
    F.col('city').alias('station_city'),
    F.col('state').alias('station_state'),
    F.col('country').alias('station_country'),
    F.col('latitude').alias('station_lat'),
    F.col('longitude').alias('station_lon'),
)
tar_dim = tariffs.select(
    'tariff_id',
    F.col('name').alias('tariff_name'),
    F.col('price_per_kwh').alias('tariff_price_per_kwh'),
    F.col('price_per_min').alias('tariff_price_per_min'),
)

pay_win = Window.partitionBy('session_id').orderBy(F.col('created_at').desc())
pay_dim = (
    payments
    .withColumn('_rn', F.row_number().over(pay_win)).filter(F.col('_rn') == 1).drop('_rn')
    .select('session_id', 'payment_id', 'amount_aud',
            F.col('status').alias('payment_status'), 'payment_method')
)

sessions_enriched = (
    sessions
    .join(cust_dim, on='customer_id', how='left')
    .join(veh_dim,  on='vehicle_id',  how='left')
    .join(chg_dim,  on='charger_id',  how='left')
    .join(sta_dim,  on='station_id',  how='left')
    .join(tar_dim,  on='tariff_id',   how='left')
    .join(pay_dim,  on='session_id',  how='left')
    .withColumn('session_date',  F.to_date('start_time'))
    .withColumn('session_year',  F.year('start_time'))
    .withColumn('session_month', F.month('start_time'))
    .withColumn('session_hour',  F.hour('start_time'))
    .withColumn('cost_aud', F.coalesce(
        F.col('amount_aud'),
        (F.col('tariff_price_per_kwh') * F.col('energy_kwh')
         + F.col('tariff_price_per_min') * F.col('duration_minutes')).cast('decimal(10,2)')
    ))
    .withColumn('cost_per_kwh', F.when(
        F.col('energy_kwh').isNotNull() & (F.col('energy_kwh') > 0),
        (F.col('cost_aud') / F.col('energy_kwh')).cast('decimal(8,4)')
    ))
    .withColumn('battery_fill_pct', F.when(
        F.col('battery_capacity_kwh').isNotNull() & (F.col('battery_capacity_kwh') > 0),
        (F.col('energy_kwh') / F.col('battery_capacity_kwh') * 100).cast('decimal(6,2)')
    ))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'sessions_enriched : {sessions_enriched.count()} rows  {len(sessions_enriched.columns)} cols')

In [ ]:
# Cell 5 — Build revenue_by_station (aggregated daily)
revenue_by_station = (
    sessions_enriched
    .groupBy('station_id', 'station_name', 'station_city', 'station_state', 'station_country', 'session_date')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('cost_aud').cast('decimal(14,2)').alias('total_revenue_aud'),
        F.avg('duration_minutes').cast('decimal(8,2)').alias('avg_duration_min'),
        F.avg('cost_per_kwh').cast('decimal(8,4)').alias('avg_cost_per_kwh'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.countDistinct('charger_id').alias('chargers_used'),
    )
    .withColumn('report_year',  F.year('session_date'))
    .withColumn('report_month', F.month('session_date'))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'revenue_by_station : {revenue_by_station.count()} rows')

In [ ]:
# Cell 6 — Build customer_summary (lifetime stats per customer)
complaint_counts = (
    complaints
    .groupBy('customer_id')
    .agg(F.count('complaint_id').alias('total_complaints'))
)

customer_summary = (
    sessions_enriched
    .groupBy('customer_id', 'customer_name', 'customer_email',
             'customer_city', 'customer_state', 'customer_country')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('cost_aud').cast('decimal(14,2)').alias('total_spend_aud'),
        F.avg('duration_minutes').cast('decimal(8,2)').alias('avg_duration_min'),
        F.min('start_time').alias('first_session_ts'),
        F.max('start_time').alias('last_session_ts'),
        F.countDistinct('station_id').alias('stations_visited'),
        F.countDistinct('vehicle_id').alias('vehicles_used'),
    )
    .join(complaint_counts, on='customer_id', how='left')
    .withColumn('total_complaints', F.coalesce('total_complaints', F.lit(0)))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'customer_summary : {customer_summary.count()} rows')

In [ ]:
# Cell 7 — Build charger_utilisation (daily usage per charger)
charger_utilisation = (
    sessions_enriched
    .groupBy('charger_id', 'charger_type', 'charger_power_kw', 'connector_type',
             'station_id', 'station_name', 'session_date')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('duration_minutes').cast('decimal(10,2)').alias('total_active_minutes'),
        F.sum('energy_kwh').cast('decimal(12,4)').alias('total_energy_kwh'),
        F.sum('cost_aud').cast('decimal(12,2)').alias('total_revenue_aud'),
        F.avg('energy_kwh').cast('decimal(8,4)').alias('avg_energy_per_session_kwh'),
        F.countDistinct('customer_id').alias('unique_customers'),
    )
    # Utilisation pct = active minutes / 1440 minutes per day
    .withColumn(
        'utilisation_pct',
        F.least(
            (F.col('total_active_minutes') / F.lit(1440) * 100).cast('decimal(6,2)'),
            F.lit(100).cast('decimal(6,2)')
        )
    )
    .withColumn('report_year',  F.year('session_date'))
    .withColumn('report_month', F.month('session_date'))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'charger_utilisation : {charger_utilisation.count()} rows')

In [ ]:
# Cell 8 — Build maintenance_summary (one row per maintenance event, enriched)
emp_dim = employees.select(
    F.col('employee_id').alias('technician_id'),
    F.concat_ws(' ', 'first_name', 'last_name').alias('technician_name'),
    F.col('role').alias('technician_role'),
    F.col('department'),
)

maintenance_summary = (
    maintenance_events
    .join(sta_dim,  on='station_id',    how='left')
    .join(chg_dim,  on='charger_id',    how='left')
    .join(emp_dim,  on='technician_id', how='left')
    .select(
        'event_id', 'charger_id', 'station_id', 'station_name',
        'station_city', 'station_state',
        'charger_type', 'charger_power_kw',
        'event_type', 'description', 'status',
        'scheduled_date', 'completed_date',
        'technician_id', 'technician_name', 'technician_role', 'department',
        # MTTR in hours (completed - scheduled)
        F.when(
            F.col('completed_date').isNotNull() & F.col('scheduled_date').isNotNull(),
            F.datediff('completed_date', 'scheduled_date') * 24
        ).cast('decimal(8,2)').alias('resolution_hours'),
        'updated_at',
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'maintenance_summary : {maintenance_summary.count()} rows')

In [ ]:
# Cell 9 — Write all 5 Gold marts (full overwrite)
marts = [
    ('sessions_enriched',   sessions_enriched,   ['session_year', 'session_month']),
    ('revenue_by_station',  revenue_by_station,  ['report_year',  'report_month']),
    ('customer_summary',    customer_summary,    None),
    ('charger_utilisation', charger_utilisation, ['report_year',  'report_month']),
    ('maintenance_summary', maintenance_summary, None),
]

results = []
for mart_name, df, partition_cols in marts:
    path = f'{GOLD_API}/{mart_name}'
    print(f'  Writing {mart_name} ...', end=' ')
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    row_count = df.count()
    results.append({'mart': mart_name, 'rows': row_count})
    print(f'OK ({row_count} rows)')

print('\nAll Gold marts written.')

In [ ]:
# Cell 10 — Run summary + spot verify
print('=' * 60)
print('GOLD API ALL MARTS v2 — RUN SUMMARY')
print('=' * 60)
print(f'  run_ts   : {RUN_TS}')
print()
for r in results:
    print(f"  {r['mart']:<25} rows={r['rows']:>8}")

print('\nSpot check — Revenue by station (top 5):')
spark.read.format('delta').load(f'{GOLD_API}/revenue_by_station') \
    .groupBy('station_name') \
    .agg(F.sum('total_revenue_aud').alias('revenue')) \
    .orderBy(F.col('revenue').desc()).show(5, truncate=False)